# 11b: Benefit by severity, under BOTH aggregation rules (EXPLORATORY)

Supersedes notebook 11, which used only the matched-workload rule.

The two rules are different objects and can disagree:

- **Union OR (true OR gate).** Refer everything either model refers. A case is missed only if
  both miss it, so **misses can only fall**; false positives rise. Baseline for fairness: the
  better constituent given the **same enlarged referral volume** (label-free top-R_or). Neither
  side gets free referrals, and the ensemble never drops a case it caught.
- **Matched workload (the locked primary).** Rank by the higher of the two models' percentiles
  and refer the top R, where R is the single model's referral count. This is a *truncated*
  union: a marginal catch of one model can be displaced by a high-confidence case from the
  other, so **misses can rise** for a given grade. Strictly it is max-score aggregation at
  fixed referral volume, not an OR gate, and the paper should call it that.

Both are reported per severity grade with patient-clustered intervals, plus the headroom
(marginal miss rate by grade) and the referral cost of the union rule.

Fresh set, 55 pairs x 9 seed combinations. Exploratory; the pre-specified primary is unchanged.
CPU runtime, about 25 minutes. Run cells 1-5.

In [1]:
# Cell 1: load fresh predictions
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive','v3')
F_PRED='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RES='1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'; F_FIG='18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'
def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def up(p,f,m):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]}, media_body=MediaFileUpload(p,mimetype=m), fields='id').execute()
    print('  uploaded', os.path.basename(p))

ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]; PAIRS=list(itertools.combinations(ARCHS,2)); COMBOS=[(i,j) for i in SEEDS for j in SEEDS]
def tag(k): return '' if k==0 else f'_seed{k}'
BURDEN=0.25; WGT={4:500,3:200,2:15}

P={}; ref=None
for a in ARCHS:
    for k in SEEDS:
        fn=f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv'
        if not os.path.exists(fn): dl(find_id(fn,F_PRED), fn)
        d=pd.read_csv(fn)
        if ref is None: ref=d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
        P[(a,k)]=d['predicted_probability'].to_numpy(np.float64)
GRD=ref['true_grade'].to_numpy(int); PID=ref['patient_id'].astype(str).to_numpy(); N=len(GRD)
PATS=pd.unique(PID); PAT_IMGS=[np.where(PID==p)[0] for p in PATS]
print(f'fresh: {N} images, {len(PATS)} patients | grades:', {g:int((GRD==g).sum()) for g in range(5)})

fresh: 15794 images, 7897 patients | grades: {0: 11578, 1: 1131, 2: 2373, 3: 384, 4: 328}


In [2]:
# Cell 2: both rules, per grade
def topR(p,R):
    m=np.zeros(len(p),bool); m[np.argsort(-p,kind='stable')[:R]]=True; return m

def analyse(idx):
    g=GRD[idx]; n=len(idx); R=int(round(BURDEN*n))
    M={gr:(g==gr) for gr in (2,3,4)}
    sub={k:p[idx] for k,p in P.items()}
    REF={k:topR(v,R) for k,v in sub.items()}
    RK={k:rankdata(v)/n for k,v in sub.items()}
    out={('matched',gr):[] for gr in (2,3,4)}; out.update({('union',gr):[] for gr in (2,3,4)})
    marg={gr:[] for gr in (2,3,4)}; extra=[]
    for a,b in PAIRS:
        for i,j in COMBOS:
            ra, rb = REF[(a,i)], REF[(b,j)]
            # choose the better constituent once, by weighted miss cost, so grades decompose one comparison
            cA=sum(WGT[gr]*((~ra)&M[gr]).sum() for gr in (2,3,4))
            cB=sum(WGT[gr]*((~rb)&M[gr]).sum() for gr in (2,3,4))
            base_ref, base_p = (ra, sub[(a,i)]) if cA<=cB else (rb, sub[(b,j)])
            # rule 1: matched workload (locked primary): top-R by max rank
            m_ref = topR(np.maximum(RK[(a,i)], RK[(b,j)]), R)
            # rule 2: true union OR; baseline given the SAME enlarged volume
            u_ref = ra | rb; R_or = int(u_ref.sum())
            base_at_Ror = topR(base_p, R_or)
            extra.append(R_or - R)
            for gr in (2,3,4):
                out[('matched',gr)].append(int(((~base_ref)&M[gr]).sum()) - int(((~m_ref)&M[gr]).sum()))
                out[('union',gr)].append(int(((~base_at_Ror)&M[gr]).sum()) - int(((~u_ref)&M[gr]).sum()))
                marg[gr].append(((~base_ref)&M[gr]).sum()/max(1,M[gr].sum()))
    return ({k:float(np.mean(v)) for k,v in out.items()},
            {gr:float(np.mean(marg[gr])) for gr in (2,3,4)}, float(np.mean(extra)))

t0=time.time(); AV, MARG, EXTRA = analyse(np.arange(N))
print(f'computed in {time.time()-t0:.0f}s | union refers {EXTRA:.0f} more patients than the single model '
      f'({EXTRA/N*100:.1f}% of the set)\n')
rows=[]
for gr,name in ((4,'grade 4 (PDR)'),(3,'grade 3 (severe NPDR)'),(2,'grade 2 (moderate NPDR)')):
    rows.append({'grade':name,'eyes':int((GRD==gr).sum()),'baseline_miss_rate':round(MARG[gr],4),
                 'avoided_matched':round(AV[('matched',gr)],2),
                 'avoided_union_same_volume':round(AV[('union',gr)],2),
                 'weighted_matched':round(AV[('matched',gr)]*WGT[gr],1),
                 'weighted_union':round(AV[('union',gr)]*WGT[gr],1)})
T=pd.DataFrame(rows); print(T.to_string(index=False))

computed in 2s | union refers 1196 more patients than the single model (7.6% of the set)

                  grade  eyes  baseline_miss_rate  avoided_matched  avoided_union_same_volume  weighted_matched  weighted_union
          grade 4 (PDR)   328              0.0416             3.20                       2.27            1602.0          1133.3
  grade 3 (severe NPDR)   384              0.0616             1.59                       1.37             317.2           274.3
grade 2 (moderate NPDR)  2373              0.3403            45.99                      47.48             689.8           712.2


In [3]:
# Cell 3: patient-clustered bootstrap for both rules
rng=np.random.default_rng(20260922); B=300; boots=[]
t0=time.time()
for b in range(B):
    idx=np.concatenate([PAT_IMGS[i] for i in rng.integers(0,len(PATS),len(PATS))])
    av,_,_=analyse(idx)
    boots.append([av[('matched',4)],av[('matched',3)],av[('matched',2)],
                  av[('union',4)],av[('union',3)],av[('union',2)]])
    if (b+1)%50==0: print(f'  {b+1}/{B} ({time.time()-t0:.0f}s)')
boots=np.array(boots)
keys=[('matched',4),('matched',3),('matched',2),('union',4),('union',3),('union',2)]
CI={f'{m}_grade{g}':[round(float(x),2) for x in np.percentile(boots[:,i],[2.5,97.5])] for i,(m,g) in enumerate(keys)}
print('\nnet misses avoided per pair, 95% patient-clustered intervals')
for m in ('matched','union'):
    print(f'  {m}:')
    for gr,name in ((4,'grade 4 (PDR)     '),(3,'grade 3 (severe)  '),(2,'grade 2 (moderate)')):
        print(f"    {name} {AV[(m,gr)]:7.2f}  {CI[f'{m}_grade{gr}']}")

  50/300 (101s)
  100/300 (199s)
  150/300 (299s)
  200/300 (402s)
  250/300 (501s)
  300/300 (601s)

net misses avoided per pair, 95% patient-clustered intervals
  matched:
    grade 4 (PDR)         3.20  [1.31, 4.15]
    grade 3 (severe)      1.59  [0.32, 2.98]
    grade 2 (moderate)   45.99  [34.43, 55.27]
  union:
    grade 4 (PDR)         2.27  [0.5, 3.74]
    grade 3 (severe)      1.37  [-0.27, 3.03]
    grade 2 (moderate)   47.48  [38.12, 58.16]


In [4]:
# Cell 4: clinical translation
npat=len(PATS)
CLIN={'screened_patients':int(npat), 'union_extra_referrals_per_1000': round(EXTRA/npat*1000,1)}
for m in ('matched','union'):
    CLIN[m]={f'grade{gr}_avoided_per_1000_screened': round(AV[(m,gr)]/npat*1000,3) for gr in (2,3,4)}
    p4=AV[(m,4)]/npat*1000
    CLIN[m]['pdr_per_100000_screened']=round(p4*100,1)
    CLIN[m]['number_needed_to_screen_per_pdr_avoided']=round(1000/p4) if p4>0 else None
CLIN['note']=('A missed PDR at one screening round is not equivalent to blindness (rescreening, '
              'symptomatic presentation, later treatment), so any QALY figure is an upper bound.')
print(json.dumps(CLIN, indent=2))
print('\nReading: the union rule is the deployment question (misses can only fall; the cost is extra')
print('referrals). The matched rule is triage under fixed capacity, where a rescue can displace a')
print('catch. If grade 4 is positive under union but not matched, capacity is the binding constraint,')
print('not model complementarity.')

{
  "screened_patients": 7897,
  "union_extra_referrals_per_1000": 151.5,
  "matched": {
    "grade2_avoided_per_1000_screened": 5.823,
    "grade3_avoided_per_1000_screened": 0.201,
    "grade4_avoided_per_1000_screened": 0.406,
    "pdr_per_100000_screened": 40.6,
    "number_needed_to_screen_per_pdr_avoided": 2465
  },
  "union": {
    "grade2_avoided_per_1000_screened": 6.012,
    "grade3_avoided_per_1000_screened": 0.174,
    "grade4_avoided_per_1000_screened": 0.287,
    "pdr_per_100000_screened": 28.7,
    "number_needed_to_screen_per_pdr_avoided": 3484
  },
  "note": "A missed PDR at one screening round is not equivalent to blindness (rescreening, symptomatic presentation, later treatment), so any QALY figure is an upper bound."
}

Reading: the union rule is the deployment question (misses can only fall; the cost is extra
referrals). The matched rule is triage under fixed capacity, where a rescue can displace a
catch. If grade 4 is positive under union but not matched, capacity

In [5]:
# Cell 5: figure and save
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
names=['Grade 4\n(PDR)','Grade 3\n(severe NPDR)','Grade 2\n(moderate NPDR)']; g=[4,3,2]
x=np.arange(3); w=0.38
fig,ax=plt.subplots(1,2,figsize=(13,4.6))
for off,m,c in ((-w/2,'matched','C0'),(w/2,'union','C3')):
    v=[AV[(m,gr)] for gr in g]
    lo=[CI[f'{m}_grade{gr}'][0] for gr in g]; hi=[CI[f'{m}_grade{gr}'][1] for gr in g]
    ax[0].bar(x+off, v, w, label=('matched workload' if m=='matched' else 'union OR, same volume'), color=c)
    ax[0].errorbar(x+off, v, yerr=[np.array(v)-np.array(lo), np.array(hi)-np.array(v)], fmt='none', ecolor='k', capsize=3)
ax[0].set_xticks(x); ax[0].set_xticklabels(names); ax[0].axhline(0,color='grey',lw=0.8)
ax[0].set_ylabel('Net misses avoided per pair'); ax[0].legend(); ax[0].set_title('Benefit by grade, both rules')
for off,m,c in ((-w/2,'matched','C0'),(w/2,'union','C3')):
    ax[1].bar(x+off, [AV[(m,gr)]*WGT[gr] for gr in g], w, color=c)
ax[1].set_xticks(x); ax[1].set_xticklabels(names); ax[1].axhline(0,color='grey',lw=0.8)
ax[1].set_ylabel('Contribution to weighted cost advantage'); ax[1].set_title('Where the severity weight lands')
fig.suptitle('Ensemble benefit by severity: true union OR vs matched workload (fresh set, exploratory)')
fig.tight_layout(); fig.savefig('benefit_by_severity_both_rules_v3.png', dpi=200)

OUT={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'status':'EXPLORATORY; supersedes notebook 11; pre-specified primary unchanged',
     'burden':BURDEN,'weights':WGT,'by_grade':T.to_dict(orient='records'),'intervals':CI,
     'union_extra_referrals_mean':round(EXTRA,1),'clinical':CLIN,
     'definitions':{'matched':'top-R by max percentile rank; truncated union; misses can rise',
                    'union':'refer if either refers; misses can only fall; baseline is the better '
                            'constituent given the same enlarged volume'}}
with open('benefit_by_severity_both_rules_v3.json','w') as f: json.dump(OUT,f,indent=2,default=str)
T.to_csv('benefit_by_severity_both_rules_v3.csv', index=False)
up('benefit_by_severity_both_rules_v3.json',F_RES,'application/json')
up('benefit_by_severity_both_rules_v3.csv',F_RES,'text/csv')
up('benefit_by_severity_both_rules_v3.png',F_FIG,'image/png')
print('saved')

  uploaded benefit_by_severity_both_rules_v3.json
  uploaded benefit_by_severity_both_rules_v3.csv
  uploaded benefit_by_severity_both_rules_v3.png
saved
